# LightGBM训练 - 最简化版本

这份代码去掉了所有复杂功能，只保留核心训练逻辑。

## 1. 导入库

In [1]:
# 基础库
import numpy as np
import pandas as pd
import polars as pl
from pathlib import Path
import gc

# LightGBM
import lightgbm as lgb
import joblib

# 内存监控
import psutil
import os

## 2. 内存监控函数（简化版）

In [2]:
def print_memory(tag=""):
    """打印当前内存使用"""
    process = psutil.Process(os.getpid())
    mem_gb = process.memory_info().rss / 1024**3
    print(f"{tag}内存: {mem_gb:.2f} GB")

snapshots = []

def snapshot(name):
    """记录内存快照"""
    process = psutil.Process(os.getpid())
    mem_gb = process.memory_info().rss / 1024**3
    snapshots.append({'name': name, 'memory_gb': mem_gb})

def show_report():
    """显示内存报告"""
    print("\n" + "="*50)
    print("内存使用报告:")
    print("="*50)
    for s in snapshots:
        print(f"  {s['name']}: {s['memory_gb']:.2f} GB")
    print("="*50)

print("✅ 内存监控已启用")

✅ 内存监控已启用


## 3. 配置（简化到最核心）

In [3]:
# 数据路径
DATA_DIR = Path('./processed_data')
TRAIN_FILE = DATA_DIR / 'training.parquet'
VALID_FILE = DATA_DIR / 'validation.parquet'
MODEL_DIR = Path('./models')

# 特征列名（79个feature + 9个lag）
FEATURES = [f"feature_{i:02d}" for i in range(79)] + [f"responder_{i}_lag_1" for i in range(9)]

# 数据采样（防止内存溢出）
N_TRAIN_DAYS = 600   # 训练集加载多少天
N_VALID_DAYS = 100   # 验证集加载多少天
N_SYMBOLS = 39      # 加载多少个股票

# 模型参数
N_TREES = 500     # 树的数量
MAX_DEPTH = 6     # 树的最大深度

print(f"特征数: {len(FEATURES)}")
print(f"数据采样:")
print(f"  训练集: {N_TRAIN_DAYS}天 x {N_SYMBOLS}股票")
print(f"  验证集: {N_VALID_DAYS}天 x {N_SYMBOLS}股票")

特征数: 88
数据采样:
  训练集: 600天 x 39股票
  验证集: 100天 x 39股票


## 4. 加载数据（最简版本）

In [4]:
# 创建模型输出目录
MODEL_DIR.mkdir(exist_ok=True)

# 记录初始内存
snapshot("开始")
print_memory("开始: ")

# ===== 加载训练数据 =====
print("\n[1/2] 加载训练数据...")
train_lazy = pl.scan_parquet(TRAIN_FILE)

# 采样：只取最后N_TRAIN_DAYS天、前N_SYMBOLS个股票
all_dates = train_lazy.select('date_id').unique().collect().to_series().to_list()
all_dates.sort()
selected_dates = all_dates[-N_TRAIN_DAYS:]  # 最后N_TRAIN_DAYS天

all_symbols = train_lazy.select('symbol_id').unique().collect().to_series().to_list()
selected_symbols = sorted(all_symbols)[:N_SYMBOLS]  # 前N_SYMBOLS个股票

print(f"  日期: {selected_dates[0]}-{selected_dates[-1]} ({len(selected_dates)}天)")
print(f"  股票: {len(selected_symbols)}个")

# 过滤数据
train_lazy = train_lazy.filter(
    pl.col('date_id').is_in(selected_dates) & 
    pl.col('symbol_id').is_in(selected_symbols)
)

# 填充缺失值并转换为pandas
needed_cols = FEATURES + ['responder_6', 'weight', 'date_id', 'symbol_id']
train_lazy = train_lazy.select(needed_cols)
train_lazy = train_lazy.with_columns([pl.col(f).fill_null(0) for f in FEATURES])

train_df = train_lazy.collect().to_pandas()

print(f"  ✓ 训练数据: {train_df.shape}")

snapshot("训练数据加载完成")
print_memory("加载训练数据后: ")

# ===== 加载验证数据 =====
print("\n[2/2] 加载验证数据...")
valid_lazy = pl.scan_parquet(VALID_FILE)

# 采样：只取最后N_VALID_DAYS天
all_dates = valid_lazy.select('date_id').unique().collect().to_series().to_list()
all_dates.sort()
selected_dates = all_dates[-N_VALID_DAYS:]  # 最后N_VALID_DAYS天

valid_lazy = valid_lazy.filter(
    pl.col('date_id').is_in(selected_dates) & 
    pl.col('symbol_id').is_in(selected_symbols)
)

valid_lazy = valid_lazy.select(needed_cols)
valid_lazy = valid_lazy.with_columns([pl.col(f).fill_null(0) for f in FEATURES])

valid_df = valid_lazy.collect().to_pandas()

print(f"  ✓ 验证数据: {valid_df.shape}")

snapshot("验证数据加载完成")
print_memory("加载验证数据后: ")

开始: 内存: 0.23 GB

[1/2] 加载训练数据...
  日期: 999-1598 (600天)
  股票: 39个
  ✓ 训练数据: (21955208, 92)
加载训练数据后: 内存: 0.58 GB

[2/2] 加载验证数据...
  ✓ 验证数据: (3716152, 92)
加载验证数据后: 内存: 1.97 GB


## 5. 准备训练数据

In [5]:
print("\n准备训练数据...")

# 提取特征、目标、权重
X_train = train_df[FEATURES]
y_train = train_df['responder_6']
w_train = train_df['weight']

X_valid = valid_df[FEATURES]
y_valid = valid_df['responder_6']
w_valid = valid_df['weight']

print(f"训练集: {X_train.shape}")
print(f"验证集: {X_valid.shape}")

# 删除原始DataFrame，释放内存
del train_df, valid_df
gc.collect()

snapshot("准备训练数据完成")
print_memory("准备完成后: ")


准备训练数据...
训练集: (21955208, 88)
验证集: (3716152, 88)
准备完成后: 内存: 2.56 GB


## 6. 定义评估指标（加权R²）

In [6]:
def weighted_r2(y_true, y_pred, weights):
    """
    加权R²得分
    
    公式: R² = 1 - Σ(w*(y-ŷ)²) / Σ(w*y²)
    """
    numerator = np.average((y_true - y_pred)**2, weights=weights)
    denominator = np.average(y_true**2, weights=weights) + 1e-38
    return 1 - numerator / denominator

# LightGBM需要的格式
def lgb_metric(y_true, y_pred, weights):
    r2 = weighted_r2(y_true, y_pred, weights)
    return 'r2', r2, True  # (名称, 值, 越大越好)

print("评估函数已定义")

评估函数已定义


## 7. 训练模型（核心代码）

In [7]:
print("\n开始训练...")
print("="*50)

snapshot("训练开始")

# 创建模型
model = lgb.LGBMRegressor(
    n_estimators=N_TREES,
    max_depth=MAX_DEPTH,
    num_leaves=64,
    learning_rate=0.05,
    objective='regression',
    metric='None',           # 使用自定义指标
    device='cpu',            # Mac上用CPU
    verbose=-1,              # 不输出训练日志
)

# 训练
model.fit(
    X_train, y_train, w_train,
    eval_set=[(X_valid, y_valid, w_valid)],
    eval_metric=lgb_metric,
    callbacks=[
        lgb.early_stopping(50),    # 50轮没进步就停止
        lgb.log_evaluation(50)     # 每50轮输出一次
    ]
)

snapshot("训练完成")
print_memory("训练后: ")


开始训练...


Training until validation scores don't improve for 50 rounds
[50]	valid_0's r2: 0.00573668
[100]	valid_0's r2: 0.00670448
[150]	valid_0's r2: 0.00710016
[200]	valid_0's r2: 0.00718342
[250]	valid_0's r2: 0.00723359
Early stopping, best iteration is:
[232]	valid_0's r2: 0.00727508
训练后: 内存: 0.68 GB


## 8. 评估结果

In [8]:
# 预测
train_pred = model.predict(X_train)
valid_pred = model.predict(X_valid)

# 计算R²
train_r2 = weighted_r2(y_train, train_pred, w_train)
valid_r2 = weighted_r2(y_valid, valid_pred, w_valid)

print("\n" + "="*50)
print("训练结果:")
print("="*50)
print(f"训练集 R²: {train_r2:.6f}")
print(f"验证集 R²: {valid_r2:.6f}")
print(f"实际训练树数: {model.booster_.num_trees()}")
print("="*50)

show_report()


训练结果:
训练集 R²: 0.035327
验证集 R²: 0.006829
实际训练树数: 232

内存使用报告:
  开始: 0.23 GB
  训练数据加载完成: 0.58 GB
  验证数据加载完成: 1.97 GB
  准备训练数据完成: 2.56 GB
  训练开始: 2.41 GB
  训练完成: 0.68 GB


## 9. 保存模型

In [9]:
# 保存模型
model_path = MODEL_DIR / 'simple_lgb_model.pkl'
joblib.dump(model, model_path)
print(f"\n模型已保存: {model_path}")

# 清理内存
del model, X_train, y_train, X_valid, y_valid
gc.collect()

print_memory("清理后: ")


模型已保存: models/simple_lgb_model.pkl
清理后: 内存: 0.87 GB


## 10. 总结

In [10]:
print(f"""
========================================
训练流程总结
========================================

1. 加载数据
   - 训练集: {N_TRAIN_DAYS}天 x {N_SYMBOLS}股票
   - 验证集: {N_VALID_DAYS}天 x {N_SYMBOLS}股票
   - 使用polars lazy loading

2. 准备数据
   - 提取特征X、目标y、权重w
   - 删除原始DataFrame

3. 训练模型
   - 最多{N_TREES}棵树
   - 最大深度{MAX_DEPTH}
   - Early stopping防止过拟合

4. 评估结果
   - 训练集R²: {train_r2:.4f}
   - 验证集R²: {valid_r2:.4f}

5. 保存模型
   - 模型文件: simple_lgb_model.pkl
========================================
""")


训练流程总结

1. 加载数据
   - 训练集: 600天 x 39股票
   - 验证集: 100天 x 39股票
   - 使用polars lazy loading

2. 准备数据
   - 提取特征X、目标y、权重w
   - 删除原始DataFrame

3. 训练模型
   - 最多500棵树
   - 最大深度6
   - Early stopping防止过拟合

4. 评估结果
   - 训练集R²: 0.0353
   - 验证集R²: 0.0068

5. 保存模型
   - 模型文件: simple_lgb_model.pkl



## 附录：如何使用训练好的模型

In [ ]:
# 加载模型
# loaded_model = joblib.load(MODEL_DIR / 'simple_lgb_model.pkl')

# 预测新数据
# predictions = loaded_model.predict(new_data[FEATURES])

print("""
使用方法:
--------
# 加载模型
model = joblib.load('models/simple_lgb_model.pkl')

# 预测
predictions = model.predict(test_data[FEATURES])

# 创建提交结果
result = pd.DataFrame({
    'row_id': test_data['row_id'],
    'responder_6': predictions
})
""")


使用方法:
--------
# 加载模型
model = joblib.load('models/simple_lgb_model.pkl')

# 预测
predictions = model.predict(test_data[FEATURES])

# 创建提交结果
result = pd.DataFrame({
    'row_id': test_data['row_id'],
    'responder_6': predictions
})



: 